# KG1 V1244 CoT-safe — TREINO vigiado (Claude)
Rota A. **Run all.** Pré-req: Colab **A100 (High-RAM)** + Secret `HF_KEY` (escrita).

**1 único knob** na célula de config: `MODE='SMOKE'` (8 steps, valida mecânica) ou `MODE='REAL'` (160 steps, treino de verdade).
- `REAL`: lr 5e-6→1e-6, NUM_EPOCHS=6 → **160 steps** (planned=31×6=186, cap 160), 4 checkpoints (40/80/120/160) + final.
- **Live-log liga sozinho** → streaming p/ HF `kg1-live-logs` (Claude monitora AO VIVO).
- O **juiz REAL** de score é o **Notebook B** (full947) — rode nele cada checkpoint pra achar o melhor.

import os, subprocess, sys
print('[1/4] clone repo branch', flush=True)
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'], check=True)
os.chdir('/content/kg1')
print('[2/4] deps base', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','bitsandbytes','safetensors','huggingface_hub','hf_xet','einops','ninja'], check=False)
import torch
py=f"cp{sys.version_info.major}{sys.version_info.minor}"
tmm='.'.join(torch.__version__.split('+')[0].split('.')[:2])      # ex: 2.8
cu='cu'+((torch.version.cuda or '12').split('.')[0])              # ex: cu12
abi='TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'
print(f'[env] {py} torch{tmm} {cu} cxx11abi{abi}', flush=True)
print('[3/4] mamba-ssm 2.3.1 (WHEEL PRONTO ~10s; fallback source ~25min)', flush=True)
url=f"https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{tmm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
print('  tentando wheel:', url, flush=True)
if subprocess.run([sys.executable,'-m','pip','install','--no-deps',url]).returncode!=0:
    print('  >>> wheel nao casou (torch fora de 2.6-2.10?) -> compilando do source', flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'], check=False)
print('[4/4] causal-conv1d 1.6.1 (source ~5min; sem wheel p/ cu12+torch2.x)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'], check=False)
# valida import (se falhar, o treino avisaria depois)
try:
    import mamba_ssm, causal_conv1d; print('IMPORT OK: mamba_ssm + causal_conv1d', flush=True)
except Exception as e:
    print('AVISO import:', str(e)[:120], flush=True)
print('DEPS OK', flush=True)


In [ ]:
import os, subprocess, sys
print('[1/3] clone repo branch', flush=True)
subprocess.run(['git','clone','--depth','1','--branch','claude/v1244-cot-safe','https://github.com/FELIPEACASTRO/KG1-NVIDIA.git','/content/kg1'], check=True)
os.chdir('/content/kg1')
print('[2/3] deps base', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6','peft==0.19.1','accelerate==1.13.0','bitsandbytes','safetensors','huggingface_hub','hf_xet'], check=False)
print('[3/3] mamba+causal (source build ~15-25min, normal)', flush=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','mamba-ssm==2.3.1'], check=False)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','causal-conv1d==1.6.1'], check=False)
print('DEPS OK', flush=True)


In [ ]:
import os, time
# ==========================================================================
#  ESCOLHA O MODO  (1 unico knob)
MODE = 'SMOKE'  # COMECE com 'SMOKE' (ensaio ~35min). Troque p/ 'REAL' (160 steps) SO apos o GO do Claude.
# ==========================================================================
os.environ['DATA_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_micro_consolidation_train.jsonl'
os.environ['VAL_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_scorelive_evalset_170.jsonl'
os.environ['INIT_ADAPTER_REPO']='felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086'
os.environ['INIT_ADAPTER_REVISION']='f4134a6d223249d27be2f1c5d94ed59d118d1ce5'
os.environ['REQUIRE_INIT_ADAPTER']='1'
os.environ['LORA_R']='32'; os.environ['LORA_ALPHA']='32'
os.environ['BATCH_SIZE']='32'            # epoch_steps = ceil(979/32) = 31
os.environ['LEARNING_RATE']='5e-6'; os.environ['FINAL_LEARNING_RATE']='1e-6'   # escada validada
os.environ['BOXED_PAYLOAD_LOSS_WEIGHT']='1.0'
os.environ['REQUIRE_OFFSET_MASK']='1'
os.environ['OUTPUT_REPO']='felipesp1983/kg1-v1244-cot-candidate'
os.environ['UPLOAD_TO_HF']='1'
if MODE=='REAL':
    # 160 steps REAIS: planned = ceil(979/32)*NUM_EPOCHS = 31*6 = 186 -> min(186,160)=160.
    # (CUIDADO: com NUM_EPOCHS=1, total=min(31,160)=31 -> rodaria so 31 steps! por isso epochs=6)
    os.environ['MAX_STEPS']='160'; os.environ['NUM_EPOCHS']='6'
    os.environ['EVAL_EVERY_STEPS']='40'; os.environ['SAVE_EVERY_STEPS']='40'   # 4 checkpoints + final
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='0'   # treino LONGO: hiccup de upload NAO aborta (retry 60s)
else:  # SMOKE
    os.environ['MAX_STEPS']='8'; os.environ['NUM_EPOCHS']='1'
    os.environ['EVAL_EVERY_STEPS']='4'; os.environ['SAVE_EVERY_STEPS']='4'
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='1'   # smoke: EXIGE streaming (valida o live-log)
print('MODE=',MODE,'| MAX_STEPS=',os.environ['MAX_STEPS'],'NUM_EPOCHS=',os.environ['NUM_EPOCHS'],
      '| lr=',os.environ['LEARNING_RATE'],'->',os.environ['FINAL_LEARNING_RATE'],
      '| eval/save@',os.environ['EVAL_EVERY_STEPS'],'| live_log_require=',os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD'],flush=True)
print('Dataset=micro 979 | INIT=086 pinado | OUTPUT=',os.environ['OUTPUT_REPO'],flush=True)
if MODE=='REAL':
    print('>>> ATENCAO: treino REAL (~2-3h, custo GPU). Confirme o GO/briefing antes de Run all. <<<',flush=True)
else:
    print('>>> SMOKE (ensaio ~35min): valida tudo + mede tempo/step. Apos OK do Claude, troque MODE=REAL. <<<',flush=True)


In [ ]:
import os, time
# ==========================================================================
#  ESCOLHA O MODO  (1 unico knob)
MODE = 'REAL'   # 'SMOKE' = 8 steps (so valida mecanica). 'REAL' = treino de verdade (160 steps).
# ==========================================================================
os.environ['DATA_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_micro_consolidation_train.jsonl'
os.environ['VAL_FILE']='/content/kg1/artifacts/v1244_cot_safe/v1244_scorelive_evalset_170.jsonl'
os.environ['INIT_ADAPTER_REPO']='felipesp1983/kg1-recovered-v291-v290-checkpoint6-submit086'
os.environ['INIT_ADAPTER_REVISION']='f4134a6d223249d27be2f1c5d94ed59d118d1ce5'
os.environ['REQUIRE_INIT_ADAPTER']='1'
os.environ['LORA_R']='32'; os.environ['LORA_ALPHA']='32'
os.environ['BATCH_SIZE']='32'            # epoch_steps = ceil(979/32) = 31
os.environ['LEARNING_RATE']='5e-6'; os.environ['FINAL_LEARNING_RATE']='1e-6'   # escada validada
os.environ['BOXED_PAYLOAD_LOSS_WEIGHT']='1.0'
os.environ['REQUIRE_OFFSET_MASK']='1'
os.environ['OUTPUT_REPO']='felipesp1983/kg1-v1244-cot-candidate'
os.environ['UPLOAD_TO_HF']='1'
if MODE=='REAL':
    # 160 steps REAIS: planned = ceil(979/32)*NUM_EPOCHS = 31*6 = 186 -> min(186,160)=160.
    # (CUIDADO: com NUM_EPOCHS=1, total=min(31,160)=31 -> rodaria so 31 steps! por isso epochs=6)
    os.environ['MAX_STEPS']='160'; os.environ['NUM_EPOCHS']='6'
    os.environ['EVAL_EVERY_STEPS']='40'; os.environ['SAVE_EVERY_STEPS']='40'   # 4 checkpoints + final
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='0'   # treino LONGO: hiccup de upload NAO aborta (retry 60s)
else:  # SMOKE
    os.environ['MAX_STEPS']='8'; os.environ['NUM_EPOCHS']='1'
    os.environ['EVAL_EVERY_STEPS']='4'; os.environ['SAVE_EVERY_STEPS']='4'
    os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD']='1'   # smoke: EXIGE streaming (valida o live-log)
print('MODE=',MODE,'| MAX_STEPS=',os.environ['MAX_STEPS'],'NUM_EPOCHS=',os.environ['NUM_EPOCHS'],
      '| lr=',os.environ['LEARNING_RATE'],'->',os.environ['FINAL_LEARNING_RATE'],
      '| eval/save@',os.environ['EVAL_EVERY_STEPS'],'| live_log_require=',os.environ['KG1_REQUIRE_LIVE_LOG_UPLOAD'],flush=True)
print('Dataset=micro 979 | INIT=086 pinado | OUTPUT=',os.environ['OUTPUT_REPO'],flush=True)
if MODE=='REAL':
    print('>>> ATENCAO: treino REAL (~2-3h, custo GPU). Confirme o GO/briefing antes de Run all. <<<',flush=True)


In [ ]:
import subprocess, sys, os, time
os.chdir('/content/kg1')
# --- LIVE-LOG: stream stdout -> HF kg1-live-logs a cada 60s (Claude monitora AO VIVO via API) ---
os.environ['KG1_LIVE_LOG_HF_REPO']='felipesp1983/kg1-live-logs'
os.environ['KG1_LIVE_LOG_HF_REPO_TYPE']='dataset'
os.environ['RUN_ID']='v1244_train_'+time.strftime('%Y%m%d_%H%M%S')
# require=1 GARANTE o streaming (valida token+repo+1o upload e aborta se streaming quebrar).
# TREINO REAL LONGO (2h+, MAX_STEPS 120-160): troque p/ '0' — assim um hiccup de upload NAO aborta
# o treino (o wrapper tem retry 60s e os logs locais sempre sao salvos).
os.environ.setdefault('KG1_REQUIRE_LIVE_LOG_UPLOAD','1')
# watchdog de stall: 45min (nao mata durante o load lento do 30B; ainda mata hang real/NaN/OOM)
os.environ.setdefault('KG1_WATCHDOG_STALE_SECONDS','2700')
print('LIVE RUN_ID=', os.environ['RUN_ID'], '-> HF:', os.environ['KG1_LIVE_LOG_HF_REPO']+'/colab/'+os.environ['RUN_ID'], flush=True)
print('   adapter sera salvo em:', os.environ['OUTPUT_REPO']+'/runs/'+os.environ['RUN_ID']+'/{final,checkpoint-N}', flush=True)
# trainer (env-driven) rodado VIA wrapper realtime: streaming + status.json + watchdog
r=subprocess.run([sys.executable,'scripts/kg1_colab_realtime_runner.py','--','python','scripts/hf_job_train_v90.py'])
print('RETURN_CODE=', r.returncode, flush=True)
print('Se RC=0 e adapter subiu p/ runs/<RUN_ID>/ -> rode o NOTEBOOK B (CAND_RUN_ID='+os.environ['RUN_ID']+') p/ ACC real no full947.', flush=True)
